In [10]:
import json
import csv
from pathlib import Path
from collections import defaultdict

In [11]:
TARGET_MODEL_ID = "llama3.1:8b"  # Ollama model
# TARGET_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
# TARGET_MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
# TARGET_MODEL_ID = "zai-org/glm-4-9b-chat-hf"
# TARGET_MODEL_ID = "internlm/internlm3-8b-instruct"
# TARGET_MODEL_ID = "mistralai/Ministral-3-8B-Instruct-2512"

In [12]:
SA_DATASET_NAMES = ["DoS", "Fuzzy", "Gear", "RPM"]

SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

# For Ollama: llama3.1:8b becomes llama3_1_8b
# For HuggingFace: split on "/" and use last part
if "/" in TARGET_MODEL_ID:
    MODEL_TAG = TARGET_MODEL_ID.split("/")[-1].replace(".", "_").replace("-", "_")
else:
    # Ollama format: replace ":" and "." with "_"
    MODEL_TAG = TARGET_MODEL_ID.replace(":", "_").replace(".", "_").replace("-", "_")

def sa_answer_path(dataset_name: str, model_tag: str) -> Path:
    return Path(f"{dataset_name}_sa_qa/llm_answers/{dataset_name.lower()}_sa_answers_{model_tag}.jsonl")


In [13]:
def load_sa_questions_map(path: Path):
    """
    Returns {qa_id: ground_truth_answer}
    Accepts .json (list) or .jsonl inputs.
    """
    if not path.exists():
        return {}
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
    qmap = {}
    for rec in data:
        qa_id = rec.get("qa_id")
        answer = rec.get("ground_truth", rec.get("answer", ""))
        if qa_id is not None:
            qmap[qa_id] = answer
    return qmap


def load_sa_answers(path: Path):
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""



In [14]:
def compute_token_f1(pred: str, gold: str) -> float:
    """Token-level F1 score: overlap of tokens between prediction and gold."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    
    pred_tokens = set(pred.split())
    gold_tokens = set(gold.split())
    
    if not pred_tokens or not gold_tokens:
        return 0.0
    
    intersection = len(pred_tokens & gold_tokens)
    precision = intersection / len(pred_tokens)
    recall = intersection / len(gold_tokens)
    
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def compute_rouge_l(pred: str, gold: str) -> float:
    """ROUGE-L: Longest Common Subsequence based F1."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    
    # Longest Common Subsequence
    m, n = len(pred_tokens), len(gold_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i-1] == gold_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    
    lcs_len = dp[m][n]
    precision = lcs_len / len(pred_tokens) if pred_tokens else 0.0
    recall = lcs_len / len(gold_tokens) if gold_tokens else 0.0
    
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

In [15]:
def compute_sa_records(qmap, answers):
    """Build evaluation records with Token F1 and ROUGE-L metrics."""
    eval_records = []
    for rec in answers:
        qa_id = rec.get("qa_id")
        if qa_id is None:
            continue
        pred_raw = rec.get("llm_answer", "")
        gold_raw = rec.get("ground_truth") or qmap.get(qa_id, "")
        pred = normalize_text(pred_raw)
        gold = normalize_text(gold_raw)
        
        # Compute metrics
        exact_match = 1.0 if pred == gold else 0.0
        token_f1 = compute_token_f1(pred, gold)
        rouge_l = compute_rouge_l(pred, gold)
        
        eval_records.append({
            "qa_id": qa_id,
            "pred_raw": pred_raw,
            "gold_raw": gold_raw,
            "pred": pred,
            "gold": gold,
            "exact_match": exact_match,
            "token_f1": token_f1,
            "rouge_l": rouge_l,
        })
    return eval_records


def accuracy_from_records(records):
    """Calculate Exact Match, Token F1, and ROUGE-L from records."""
    if not records:
        return {"exact_match": 0.0, "token_f1": 0.0, "rouge_l": 0.0, "total": 0}
    
    total = len(records)
    exact_match = sum(r["exact_match"] for r in records) / total if total else 0.0
    token_f1 = sum(r["token_f1"] for r in records) / total if total else 0.0
    rouge_l = sum(r["rouge_l"] for r in records) / total if total else 0.0
    
    return {
        "exact_match": exact_match,
        "token_f1": token_f1,
        "rouge_l": rouge_l,
        "total": total
    }


In [16]:
OUTPUT_DIR = Path("LLM_SA_Performance")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_csv = OUTPUT_DIR / f"Performance_SA_{MODEL_TAG}.csv"

print(f"[DEBUG] Current working directory: {Path.cwd()}")
print(f"[DEBUG] TARGET_MODEL_ID: {TARGET_MODEL_ID}")
print(f"[DEBUG] MODEL_TAG: {MODEL_TAG}")
print(f"[DEBUG] Output CSV: {out_csv}")

header = ["Attack", "Total", "Exact Match", "Token F1", "ROUGE-L"]
all_records = []

with out_csv.open("w", encoding="utf-8", newline="") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(header)

    for name in SA_DATASET_NAMES:
        q_path = SA_QUESTION_FILES[name]
        a_path = sa_answer_path(name, MODEL_TAG)
        
        print(f"[DEBUG] Processing {name}:")
        print(f"  Questions path: {q_path} -> exists: {q_path.exists()}")
        print(f"  Answers path: {a_path} -> exists: {a_path.exists()}")

        qmap = load_sa_questions_map(q_path)
        ans_records = load_sa_answers(a_path)
        
        print(f"  Questions loaded: {len(qmap)}")
        print(f"  Answers loaded: {len(ans_records)}")

        if not qmap:
            print(f"[WARN] Skip {name}: questions missing at {q_path}.")
            continue
        if not ans_records:
            print(f"[WARN] Skip {name}: answers missing at {a_path}.")
            continue

        print(f"[INFO] Evaluating {name}...")
        try:
            eval_records = compute_sa_records(qmap, ans_records)
            print(f"  Eval records created: {len(eval_records)}")
            
            if not eval_records:
                print(f"[ERROR] No eval records generated for {name}")
                continue
            
            metrics = accuracy_from_records(eval_records)
            print(f"  Metrics: {metrics}")
            
            row = [name, metrics["total"]]
            row.extend([f"{metrics[m]:.3f}" for m in ["exact_match", "token_f1", "rouge_l"]])
            print(f"  Writing row: {row}")
            writer.writerow(row)

            all_records.extend(eval_records)
        except Exception as e:
            print(f"[ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if all_records:
        print(f"[DEBUG] Total records across all datasets: {len(all_records)}")
        metrics = accuracy_from_records(all_records)
        row = ["Combined", metrics["total"]]
        row.extend([f"{metrics[m]:.3f}" for m in ["exact_match", "token_f1", "rouge_l"]])
        writer.writerow(row)
    else:
        print(f"[WARN] No all_records to write combined metrics")

print(f"[INFO] SA performance saved to {out_csv}")


[DEBUG] Current working directory: c:\Users\Dub\Documents\GitHub\CAN_QA\QA\Create_QA_ShortAnswer_Dataset
[DEBUG] TARGET_MODEL_ID: llama3.1:8b
[DEBUG] MODEL_TAG: llama3_1_8b
[DEBUG] Output CSV: LLM_SA_Performance\Performance_SA_llama3_1_8b.csv
[DEBUG] Processing DoS:
  Questions path: DoS_sa_qa\questions\dos_sa_questions.json -> exists: True
  Answers path: DoS_sa_qa\llm_answers\dos_sa_answers_llama3_1_8b.jsonl -> exists: True
  Questions loaded: 916
  Answers loaded: 916
[INFO] Evaluating DoS...
  Eval records created: 916
  Metrics: {'exact_match': 0.06550218340611354, 'token_f1': 0.07948378041172802, 'rouge_l': 0.07948378041172802, 'total': 916}
  Writing row: ['DoS', 916, '0.066', '0.079', '0.079']
[DEBUG] Processing Fuzzy:
  Questions path: Fuzzy_sa_qa\questions\fuzzy_sa_questions.json -> exists: True
  Answers path: Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_llama3_1_8b.jsonl -> exists: True
  Questions loaded: 959
  Answers loaded: 959
[INFO] Evaluating Fuzzy...
  Eval records crea